# MLflow Configuration Notebook — Stable SQLite Setup

Run this notebook **before the forecasting training notebook**.

This version is designed for the current project layout on macOS/iCloud Drive.

It will:

1. Verify Python 3.12
2. Verify the MLflow installation
3. Keep the MLflow SQLite database **outside iCloud**
4. Keep MLflow artifacts inside the project `mlruns/` directory
5. Create the `flood_risk_forecasting` experiment
6. Test SQLite write access
7. Log a real MLflow connection-test run
8. Read the run back
9. Show recent experiment runs

Expected database location:

```text
~/.mlflow/flood-risk/mlflow.db
```

Expected artifacts location:

```text
flood-risk-ml-intoto/mlruns/
```


In [1]:
import sys
from pathlib import Path

print("=" * 80)
print("PYTHON ENVIRONMENT")
print("=" * 80)

print("Python version   :", sys.version)
print("Python executable:", sys.executable)
print("Working directory:", Path.cwd().resolve())

major, minor = sys.version_info[:2]

if (major, minor) != (3, 12):
    raise RuntimeError(
        f"This project expects Python 3.12, but the active kernel is "
        f"Python {major}.{minor}. Select the project Python 3.12 venv kernel."
    )

print()
print("Python 3.12 check: PASSED")


PYTHON ENVIRONMENT
Python version   : 3.12.13 (main, Mar  3 2026, 12:39:30) [Clang 21.0.0 (clang-2100.0.123.102)]
Python executable: /Users/charithagadari/Library/Mobile Documents/com~apple~CloudDocs/flood-risk-ml-intoto/venv/bin/python
Working directory: /Users/charithagadari/Library/Mobile Documents/com~apple~CloudDocs/flood-risk-ml-intoto/notebooks

Python 3.12 check: PASSED


In [2]:
try:
    import mlflow
except ImportError as exc:
    raise ImportError(
        "MLflow is not installed in the active notebook kernel. "
        "Activate the project venv and install mlflow==3.3.2."
    ) from exc

print("=" * 80)
print("MLFLOW PACKAGE")
print("=" * 80)

print("MLflow version :", mlflow.__version__)
print("MLflow location:", mlflow.__file__)

if mlflow.__version__ != "3.3.2":
    print()
    print(
        "WARNING: This notebook was validated for MLflow 3.3.2. "
        f"Current version is {mlflow.__version__}."
    )
else:
    print()
    print("MLflow version check: PASSED")


MLFLOW PACKAGE
MLflow version : 3.3.2
MLflow location: /Users/charithagadari/Library/Mobile Documents/com~apple~CloudDocs/flood-risk-ml-intoto/venv/lib/python3.12/site-packages/mlflow/__init__.py

MLflow version check: PASSED


In [3]:
try:
    from mlflow.store.tracking.sqlalchemy_store import SqlAlchemyStore
except ModuleNotFoundError as exc:
    raise RuntimeError(
        "SQLAlchemyStore is unavailable in the active MLflow installation.\n\n"
        "Run these commands in the project terminal:\n"
        "python -m pip uninstall -y mlflow mlflow-skinny\n"
        "python -m pip install --no-cache-dir mlflow==3.3.2"
    ) from exc

print("SQLAlchemyStore import: PASSED")


SQLAlchemyStore import: PASSED


In [4]:
# Resolve the project root.
#
# Supported notebook locations:
#   project_root/mlflow_configuration_v2.ipynb
#   project_root/notebooks/mlflow_configuration_v2.ipynb

CURRENT_DIR = Path.cwd().resolve()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

# SQLite backend OUTSIDE iCloud Drive
MLFLOW_HOME = Path.home() / ".mlflow" / "flood-risk"
MLFLOW_HOME.mkdir(parents=True, exist_ok=True)

MLFLOW_DB = MLFLOW_HOME / "mlflow.db"

# Artifacts remain inside the project
MLFLOW_ARTIFACT_DIR = PROJECT_ROOT / "mlruns"
MLFLOW_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("PROJECT AND MLFLOW PATHS")
print("=" * 80)

print("Current directory :", CURRENT_DIR)
print("Project root      :", PROJECT_ROOT)
print("MLflow home       :", MLFLOW_HOME)
print("MLflow database   :", MLFLOW_DB)
print("Artifact directory:", MLFLOW_ARTIFACT_DIR)

if "Mobile Documents" in str(MLFLOW_DB):
    raise RuntimeError(
        "The MLflow SQLite database resolved inside iCloud Drive. "
        "The database must remain outside iCloud."
    )

print()
print("MLflow database location check: PASSED")


PROJECT AND MLFLOW PATHS
Current directory : /Users/charithagadari/Library/Mobile Documents/com~apple~CloudDocs/flood-risk-ml-intoto/notebooks
Project root      : /Users/charithagadari/Library/Mobile Documents/com~apple~CloudDocs/flood-risk-ml-intoto
MLflow home       : /Users/charithagadari/.mlflow/flood-risk
MLflow database   : /Users/charithagadari/.mlflow/flood-risk/mlflow.db
Artifact directory: /Users/charithagadari/Library/Mobile Documents/com~apple~CloudDocs/flood-risk-ml-intoto/mlruns

MLflow database location check: PASSED


In [5]:
# Configure MLflow with the SQLite backend.

TRACKING_URI = f"sqlite:///{MLFLOW_DB}"

mlflow.set_tracking_uri(TRACKING_URI)

EXPERIMENT_NAME = "flood_risk_forecasting"

client = mlflow.MlflowClient()

experiment = client.get_experiment_by_name(EXPERIMENT_NAME)

if experiment is None:
    experiment_id = client.create_experiment(
        name=EXPERIMENT_NAME,
        artifact_location=MLFLOW_ARTIFACT_DIR.as_uri(),
    )
    experiment = client.get_experiment(experiment_id)

mlflow.set_experiment(EXPERIMENT_NAME)

print("=" * 80)
print("MLFLOW CONFIGURATION")
print("=" * 80)

print("Tracking URI    :", mlflow.get_tracking_uri())
print("Experiment name :", experiment.name)
print("Experiment ID   :", experiment.experiment_id)
print("Artifact URI    :", experiment.artifact_location)
print("Lifecycle stage :", experiment.lifecycle_stage)

assert str(MLFLOW_DB) in mlflow.get_tracking_uri(), (
    "MLflow is not using the expected SQLite database."
)

print()
print("MLflow configuration: PASSED")


2026/07/04 15:19:06 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/07/04 15:19:06 INFO mlflow.store.db.utils: Updating database tables
INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
INFO  [alembic.runtime.migration] Running upgrade  -> 451aebb31d03, add metric step
INFO  [alembic.runtime.migration] Running upgrade 451aebb31d03 -> 90e64c465722, migrate user column to tags
INFO  [alembic.runtime.migration] Running upgrade 90e64c465722 -> 181f10493468, allow nulls for metric values
INFO  [alembic.runtime.migration] Running upgrade 181f10493468 -> df50e92ffc5e, Add Experiment Tags Table
INFO  [alembic.runtime.migration] Running upgrade df50e92ffc5e -> 7ac759974ad8, Update run tags with larger limit
INFO  [alembic.runtime.migration] Running upgrade 7ac759974ad8 -> 89d4b8295536, create latest metrics table
INFO  [89d4b8295536_create_latest_metrics_table_py] Migration complete!
INFO  

MLFLOW CONFIGURATION
Tracking URI    : sqlite:////Users/charithagadari/.mlflow/flood-risk/mlflow.db
Experiment name : flood_risk_forecasting
Experiment ID   : 1
Artifact URI    : file:///Users/charithagadari/Library/Mobile%20Documents/com~apple~CloudDocs/flood-risk-ml-intoto/mlruns
Lifecycle stage : active

MLflow configuration: PASSED


In [6]:
# Direct SQLite write test.
#
# This confirms that the database location is writable before MLflow
# attempts to create a run.

import sqlite3

print("=" * 80)
print("SQLITE WRITE TEST")
print("=" * 80)

connection = sqlite3.connect(str(MLFLOW_DB))

try:
    cursor = connection.cursor()

    cursor.execute(
        '''
        CREATE TABLE IF NOT EXISTS mlflow_write_test (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            test_value TEXT NOT NULL
        )
        '''
    )

    cursor.execute(
        '''
        INSERT INTO mlflow_write_test (test_value)
        VALUES (?)
        ''',
        ("write_test_success",),
    )

    connection.commit()

    cursor.execute(
        '''
        SELECT test_value
        FROM mlflow_write_test
        ORDER BY id DESC
        LIMIT 1
        '''
    )

    result = cursor.fetchone()

finally:
    connection.close()

print("Database:", MLFLOW_DB)
print("Result  :", result)

assert result is not None
assert result[0] == "write_test_success"

print()
print("SQLite write test: PASSED")


SQLITE WRITE TEST
Database: /Users/charithagadari/.mlflow/flood-risk/mlflow.db
Result  : ('write_test_success',)

SQLite write test: PASSED


In [7]:
# Create a real MLflow test run.

TEST_RUN_NAME = "mlflow_connection_test"

with mlflow.start_run(
    experiment_id=experiment.experiment_id,
    run_name=TEST_RUN_NAME,
) as run:

    mlflow.log_param("purpose", "connection_test")
    mlflow.log_param("project", "flood_risk_forecasting")
    mlflow.log_param("backend", "sqlite")
    mlflow.log_param("database_location", "outside_icloud")

    mlflow.log_metric("test_mae", 0.025)
    mlflow.log_metric("test_rmse", 0.051)
    mlflow.log_metric("test_r2", 0.839)

    RUN_ID = run.info.run_id
    RUN_EXPERIMENT_ID = run.info.experiment_id

print("=" * 80)
print("TEST RUN CREATED")
print("=" * 80)

print("Run name     :", TEST_RUN_NAME)
print("Run ID       :", RUN_ID)
print("Experiment ID:", RUN_EXPERIMENT_ID)

print()
print("MLflow test run creation: PASSED")


TEST RUN CREATED
Run name     : mlflow_connection_test
Run ID       : f12d8164bd224d4395fa759d34f7552f
Experiment ID: 1

MLflow test run creation: PASSED


In [8]:
# Read the run back from MLflow.

saved_run = client.get_run(RUN_ID)

print("=" * 80)
print("TEST RUN VERIFICATION")
print("=" * 80)

print("Status :", saved_run.info.status)
print("Params :", saved_run.data.params)
print("Metrics:", saved_run.data.metrics)

assert saved_run.info.status == "FINISHED"
assert saved_run.data.params["purpose"] == "connection_test"
assert saved_run.data.params["backend"] == "sqlite"
assert saved_run.data.metrics["test_mae"] == 0.025

print()
print("MLflow read-back verification: PASSED")


TEST RUN VERIFICATION
Status : FINISHED
Params : {'purpose': 'connection_test', 'project': 'flood_risk_forecasting', 'backend': 'sqlite', 'database_location': 'outside_icloud'}
Metrics: {'test_mae': 0.025, 'test_rmse': 0.051, 'test_r2': 0.839}

MLflow read-back verification: PASSED


In [9]:
# List all experiments visible in this backend.

experiments = mlflow.search_experiments()

print("=" * 80)
print("AVAILABLE EXPERIMENTS")
print("=" * 80)

for exp in experiments:
    print(
        f"ID={exp.experiment_id:<5} "
        f"Name={exp.name:<35} "
        f"Stage={exp.lifecycle_stage}"
    )


AVAILABLE EXPERIMENTS
ID=1     Name=flood_risk_forecasting              Stage=active
ID=0     Name=Default                             Stage=active


In [10]:
# Show recent runs from the forecasting experiment.

runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["start_time DESC"],
    max_results=10,
)

display_columns = [
    column
    for column in [
        "run_id",
        "tags.mlflow.runName",
        "status",
        "metrics.test_mae",
        "metrics.test_rmse",
        "metrics.test_r2",
        "params.purpose",
        "params.backend",
    ]
    if column in runs.columns
]

runs[display_columns]


,run_id,tags.mlflow.runName,status,metrics.test_mae,metrics.test_rmse,metrics.test_r2,params.purpose,params.backend
0,f12d8164bd224d4395fa759d34f7552f,mlflow_connection_test,FINISHED,0.025,0.051,0.839,connection_test,sqlite


## Start the MLflow UI

After all cells above pass, open a terminal and run:

```bash
cd "$HOME/Library/Mobile Documents/com~apple~CloudDocs/flood-risk-ml-intoto"

source venv/bin/activate

mlflow ui \
  --backend-store-uri "sqlite:////Users/charithagadari/.mlflow/flood-risk/mlflow.db" \
  --host 127.0.0.1 \
  --port 5000
```

Then open:

```text
http://127.0.0.1:5000
```

Go to:

```text
Experiments
→ flood_risk_forecasting
→ mlflow_connection_test
```

Do not run the full forecasting notebook until the test run is visible.


In [11]:
print("=" * 80)
print("MLFLOW SETUP COMPLETE")
print("=" * 80)

print("Project root :", PROJECT_ROOT)
print("Database     :", MLFLOW_DB)
print("Artifacts    :", MLFLOW_ARTIFACT_DIR)
print("Experiment   :", EXPERIMENT_NAME)
print("Test run     :", TEST_RUN_NAME)

print()
print("Next step:")
print("1. Start the MLflow UI.")
print("2. Confirm mlflow_connection_test is visible.")
print("3. Use the same TRACKING_URI in the forecasting notebook.")

print("=" * 80)


MLFLOW SETUP COMPLETE
Project root : /Users/charithagadari/Library/Mobile Documents/com~apple~CloudDocs/flood-risk-ml-intoto
Database     : /Users/charithagadari/.mlflow/flood-risk/mlflow.db
Artifacts    : /Users/charithagadari/Library/Mobile Documents/com~apple~CloudDocs/flood-risk-ml-intoto/mlruns
Experiment   : flood_risk_forecasting
Test run     : mlflow_connection_test

Next step:
1. Start the MLflow UI.
2. Confirm mlflow_connection_test is visible.
3. Use the same TRACKING_URI in the forecasting notebook.


Now you can run forecasting notebook